# 03 · Experiments

The main experiment: every model on every dataset, first on clean training data, then under each kind and level of damage. All eight datasets run with exactly the same procedure, settings and engines.

**Per fold**, for each of the 10 settings (clean + 3 damages × 3 levels):

1. Split the 280 training rows into a learning part and a checking part (stratified 5-fold, 5 seeds = 25 folds).
2. Damage the **learning part only**; the checking part always stays clean.
3. Tune every model on the learning part with an inner 3-fold cross-validation (ROC-AUC).
4. Train the tuned model on the whole learning part.
5. Score it on the clean checking part (accuracy, macro-F1, ROC-AUC).

The quantum kernel is scored twice: at every fixed bandwidth c (only C tuned), and with c and C tuned together — the second is the quantum model that is compared with the classical ones. The 70 test rows are not used here.

**Output** (`results/runs/`): one fold-score file and one settings file per dataset. Nothing else is saved; every table and figure is computed from these files in `04_analysis`.

**Run time**: about 3–4 hours for all eight. Keep the laptop plugged in and sleep switched off. The run can be stopped at any time: finished folds are kept, and running the cell again continues where it stopped.

## 0 · Setup

In [2]:
import json, hashlib, platform, time
import numpy as np
import pandas as pd
import sklearn
from sklearn.svm import SVC
from sklearn.model_selection import StratifiedKFold, GridSearchCV
from sklearn.metrics import get_scorer

from paths import PROCESSED, RESULTS
from quantum_kernel import load_ready, get_states, kernel_from_states, off_diagonal_stats
from classical_models import get_model, SEEDS, SCORING
from corruption import LEVELS, corrupt_training, corruption_seed, noise_columns

DATASETS = ["phoneme", "electricity", "magic_telescope", "concrete", "water_potability",
            "german_credit", "steel_faults", "bank_marketing"]          # fastest first
BANDWIDTHS = [0.01, 0.025, 0.05, 0.1, 0.25, 0.5, 0.75, 1.0]
C_GRID = [0.1, 1, 10]
CLASSICAL = ["svm_rbf", "random_forest", "logistic_regression"]
SETTINGS = [("clean", 0.0)] + [(kind, level) for kind, levels in LEVELS.items() for level in levels]

OUT = RESULTS / "runs"
OUT.mkdir(parents=True, exist_ok=True)
SCORERS = {k: get_scorer(v) for k, v in SCORING.items()}
print(f"{len(DATASETS)} datasets x {len(SETTINGS)} settings x {len(SEEDS) * 5} folds -> {OUT}")

8 datasets x 10 settings x 25 folds -> ./results\runs


## 1 · One fold

**Kernels.** Clean data and wrong labels leave the features unchanged, so their kernels are slices of the clean kernel. Missing values and noise change the learning rows, so those rows are encoded again; the checking rows keep their clean states.

**Quantum model.** For every c, the inner loop picks C; the c with the best inner score is the tuned quantum model. The kernel's mean and spread on the learning rows are stored for the concentration analysis.

**Classical models.** Each model searches its own grid with the same inner folds.

In [3]:
def fold_kernels(X_learn, check, kind, q, clean_states, clean_kernels, learn):
    if kind in ("clean", "labels"):
        return {c: (clean_kernels[c][np.ix_(learn, learn)], clean_kernels[c][np.ix_(check, learn)]) for c in BANDWIDTHS}
    kernels = {}
    for c in BANDWIDTHS:
        S = get_states(X_learn, q, c=c)
        kernels[c] = (kernel_from_states(S, S), kernel_from_states(clean_states[c][check], S))
    return kernels


def quantum_fold(kernels, y_learn, y_check, seed):
    inner = list(StratifiedKFold(3, shuffle=True, random_state=seed).split(np.zeros(len(y_learn)), y_learn))
    rows, best = [], (-np.inf, None, None, None)
    for c, (K, K_check) in kernels.items():
        best_C = (-np.inf, None)
        for C in C_GRID:
            auc = np.mean([SCORERS["auc"](SVC(kernel="precomputed", C=C, class_weight="balanced")
                                          .fit(K[np.ix_(a, a)], y_learn[a]), K[np.ix_(b, a)], y_learn[b])
                           for a, b in inner])
            if auc > best_C[0]:
                best_C = (auc, C)
        model = SVC(kernel="precomputed", C=best_C[1], class_weight="balanced").fit(K, y_learn)
        scores = {k: SCORERS[k](model, K_check, y_check) for k in SCORING}
        mean, var = off_diagonal_stats(K)
        rows.append({"model": "quantum (c fixed)", "c": c, "params": f"C={best_C[1]}", "inner_auc": best_C[0],
                     "kernel_mean": mean, "kernel_var": var, **scores})
        if best_C[0] > best[0]:
            best = (best_C[0], c, best_C[1], scores)
    rows.append({"model": "quantum (c,C tuned)", "c": best[1], "params": f"c={best[1]}, C={best[2]}",
                 "inner_auc": best[0], **best[3]})
    return rows


def classical_fold(X_learn, y_learn, X_check, y_check, seed):
    rows = []
    for name in CLASSICAL:
        model, grid = get_model(name, seed)
        search = GridSearchCV(model, grid, cv=StratifiedKFold(3, shuffle=True, random_state=seed),
                              scoring="roc_auc", n_jobs=-1).fit(X_learn, y_learn)
        rows.append({"model": name, "params": str(search.best_params_), "inner_auc": search.best_score_,
                     **{k: SCORERS[k](search, X_check, y_check) for k in SCORING}})
    return rows

## 2 · One dataset

Runs all 10 settings × 25 folds and appends each finished fold to `<dataset>_folds.csv`. Before starting, it records a fingerprint of the training file; if the data file has changed since an earlier partial run, it refuses to continue instead of mixing two versions.

In [4]:
def fingerprint(name):
    return hashlib.sha1((PROCESSED / f"{name}_train_ready.csv").read_bytes()).hexdigest()[:12]


def run_dataset(name):
    start = time.perf_counter()
    X, y, _, _ = load_ready(name)
    y, q, noise_cols = y.astype(int), X.shape[1], noise_columns(name)
    fold_file, config_file = OUT / f"{name}_folds.csv", OUT / f"{name}_config.json"

    done = set()
    if fold_file.exists():
        if json.loads(config_file.read_text())["train_file_sha1"] != fingerprint(name):
            raise RuntimeError(f"{fold_file.name} was made from a different data file - move it away first.")
        old = pd.read_csv(fold_file)
        done = set(zip(old.kind, old.level.round(4), old.seed, old.fold))
    config_file.write_text(json.dumps({
        "dataset": name, "train_file_sha1": fingerprint(name), "rows": int(len(X)), "qubits": q,
        "columns_with_noise": int(noise_cols.sum()), "bandwidths": BANDWIDTHS, "C_grid": C_GRID,
        "seeds": SEEDS, "outer_folds": 5, "inner_folds": 3, "metrics": SCORING,
        "corruption_levels": {k: list(v) for k, v in LEVELS.items()},
        "versions": {"python": platform.python_version(), "numpy": np.__version__, "scikit-learn": sklearn.__version__}},
        indent=2))

    clean_states = {c: get_states(X, q, c=c) for c in BANDWIDTHS}
    clean_kernels = {c: kernel_from_states(S, S) for c, S in clean_states.items()}

    for kind, level in SETTINGS:
        t, new = time.perf_counter(), 0
        for seed in SEEDS:
            for fold, (learn, check) in enumerate(StratifiedKFold(5, shuffle=True, random_state=seed).split(X, y)):
                if (kind, round(level, 4), seed, fold) in done:
                    continue
                if kind == "clean":
                    X_learn, y_learn = X[learn].copy(), y[learn].copy()
                else:
                    X_learn, y_learn = corrupt_training(X[learn], y[learn], kind, level,
                                                        corruption_seed(name, kind, seed, fold), noise_cols)
                kernels = fold_kernels(X_learn, check, kind, q, clean_states, clean_kernels, learn)
                rows = (quantum_fold(kernels, y_learn, y[check], seed)
                        + classical_fold(X_learn, y_learn, X[check], y[check], seed))
                info = {"dataset": name, "kind": kind, "level": level, "seed": seed, "fold": fold,
                        "labels_flipped": int((y_learn != y[learn]).sum())}
                pd.DataFrame([{**info, **r} for r in rows]).to_csv(fold_file, mode="a", header=not fold_file.exists(),
                                                                   index=False)
                new += 1
        print(f"   {kind:8s} {level:<5}  {new:2d} new folds  ({time.perf_counter() - t:5.0f} s)")
    print(f"   finished in {(time.perf_counter() - start) / 60:.1f} min")

## 3 · Run all eight datasets

This is the long cell. Progress is printed after every setting.

In [5]:
total = time.perf_counter()
for name in DATASETS:
    print(name.upper())
    run_dataset(name)
print(f"\nALL DONE in {(time.perf_counter() - total) / 3600:.1f} h")

PHONEME
   clean    0.0     0 new folds  (    0 s)
   missing  0.1     0 new folds  (    0 s)
   missing  0.2     0 new folds  (    0 s)
   missing  0.3     0 new folds  (    0 s)
   noise    0.05    0 new folds  (    0 s)
   noise    0.15    0 new folds  (    0 s)
   noise    0.3     0 new folds  (    0 s)
   labels   0.05    0 new folds  (    0 s)
   labels   0.1     0 new folds  (    0 s)
   labels   0.15    0 new folds  (    0 s)
   finished in 0.0 min
ELECTRICITY
   clean    0.0     0 new folds  (    0 s)
   missing  0.1     0 new folds  (    0 s)
   missing  0.2     0 new folds  (    0 s)
   missing  0.3     0 new folds  (    0 s)
   noise    0.05    0 new folds  (    0 s)
   noise    0.15    0 new folds  (    0 s)
   noise    0.3     0 new folds  (    0 s)
   labels   0.05    0 new folds  (    0 s)
   labels   0.1     0 new folds  (    0 s)
   labels   0.15    0 new folds  (    0 s)
   finished in 0.0 min
MAGIC_TELESCOPE
   clean    0.0     0 new folds  (    0 s)
   missing  0.1

## 4 · Completeness check

Every dataset must have 250 folds (10 settings × 25), no repeated fold and no missing score, and its file must come from the current data.

In [6]:
check = {}
for name in DATASETS:
    f = pd.read_csv(OUT / f"{name}_folds.csv")
    folds = f.groupby(["kind", "level", "seed", "fold"]).ngroups
    repeats = int(f.duplicated(subset=["kind", "level", "seed", "fold", "model", "c"]).sum())
    missing = int(f[f.model != "quantum (c fixed)"].auc.isna().sum())
    same_data = json.loads((OUT / f"{name}_config.json").read_text())["train_file_sha1"] == fingerprint(name)
    check[name] = {"folds": f"{folds}/250", "repeated rows": repeats, "missing scores": missing,
                   "current data": same_data, "OK": folds == 250 and repeats == 0 and missing == 0 and same_data}

pd.DataFrame(check).T

,folds,repeated rows,missing scores,current data,OK
phoneme,250/250,0,0,True,True
electricity,250/250,0,0,True,True
magic_telescope,250/250,0,0,True,True
concrete,250/250,0,0,True,True
water_potability,250/250,0,0,True,True
german_credit,250/250,0,0,True,True
steel_faults,250/250,0,0,True,True
bank_marketing,250/250,0,0,True,True
